In [3]:
%pip install pandas numpy scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


Loads pandas library.

Pandas helps us work with tables (like Excel sheets).

✅ read_csv() loads your dataset

✅ df = DataFrame (table of data)

✅ head() shows first 5 rows

In [4]:
import pandas as pd

# load dataset
df = pd.read_csv("final_sensor_dataset.csv")

print(df.head())


       timestamp  PM2.5  PM10  TEMP   humidity  gasValue        AQI       CACI
0  3/1/2013 0:00    4.0   4.0  -0.7  24.008842     115.3  16.666667  60.059977
1  3/1/2013 1:00    8.0   8.0  -1.1  26.013678     115.3  33.333333  59.117950
2  3/1/2013 2:00    7.0   7.0  -1.1  26.013678     114.9  29.166667  59.604061
3  3/1/2013 3:00    6.0   6.0  -1.4  24.007753     116.0  25.000000  59.087210
4  3/1/2013 4:00    3.0   3.0  -2.0  24.876783     116.4  12.500000  60.980058



PART 3: Convert Timestamp to Date-Time
 - 👉 Why?

Your timestamp is text:

2013-03-01 10:00:00


AI cannot understand text time.

So we convert it into date-time format.


PART 4: Extract Time Features


👉 What this creates:
From:

2013-03-01 10:00
We get:

hour = 10
day = 1
weekday = 4

👉 Why important?

Pollution depends on time:

🚗 traffic hours

🏭 workday vs weekend

🌙 night pollution trapping

This helps AI learn patterns.





In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
# df['minute'] = df['timestamp'].dt.minute
df['weekday'] = df['timestamp'].dt.weekday


PART 5: Create Future Prediction Columns

👉 MOST IMPORTANT STEP

This tells AI:

➡ what happens in the next hour.

Example:

AQI    now	AQI_next

80	   95

95	   110

AI learns:

👉 how AQI changes.


In [6]:
df['AQI_next'] = df['AQI'].shift(-1)
df['CACI_next'] = df['CACI'].shift(-1)
# df['AQI_next'] = df['AQI'].shift(-60)
# df['CACI_next'] = df['CACI'].shift(-60)


PART 6: Remove Empty Row

In [7]:
df = df.dropna()


PART 7: Select Features (Inputs)

👉 These are inputs to AI:

Pollution + weather + time.

Think:

➡ causes of pollution.

👉 X = input data for AI

👉 y = what we want to predict

🎯 AQI after 1 hour

🎯 CACI after 1 hour

In [8]:
features = [
    'PM2.5',
    'PM10',
    'TEMP',
    'humidity',
    'gasValue',
    'hour',
    'weekday'
]

df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.weekday

df['AQI_next'] = df['AQI'].shift(-1)
df['CACI_next'] = df['CACI'].shift(-1)

df = df.dropna()

X = df[features]

y_aqi = df['AQI_next']
y_caci = df['CACI_next']



PART 8: Split Data into Training & Testing
 - Loads tool to split data.

👉 What happens:

80% data → training

20% data → testing

👉 Why?

We test if AI learned correctly.

Think:

📚 study → training

📝 exam → testing



In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_aqi, y_test_aqi = train_test_split(
    X, y_aqi, test_size=0.2, random_state=42
)

_, _, y_train_caci, y_test_caci = train_test_split(
    X, y_caci, test_size=0.2, random_state=42
)

PART 9: Train the Model

We import the AI model.

👉 Why Random Forest?

✔ accurate

✔ beginner friendly

✔ works well with pollution data

👉 What happens here:

AI studies patterns between:

inputs → next AQI

This is the learning step.

In [10]:
from sklearn.ensemble import RandomForestRegressor

# AQI model
aqi_model = RandomForestRegressor(n_estimators=200, random_state=42)
aqi_model.fit(X_train, y_train_aqi)

# CACI model
caci_model = RandomForestRegressor(n_estimators=200, random_state=42)
caci_model.fit(X_train, y_train_caci)

print("Training completed ✅")


Training completed ✅


PART 10: Check Accuracy
 - Tool to measure prediction error.
 - AI predicts values for test data.

 👉 What is error?

Difference between:

predicted vs actual.

Example:

Actual AQI = 150

Predicted AQI = 140

Error = 10

✔ smaller error = better model


In [11]:
from sklearn.metrics import mean_absolute_error

aqi_pred = aqi_model.predict(X_test)
caci_pred = caci_model.predict(X_test)

print("AQI Error:", mean_absolute_error(y_test_aqi, aqi_pred))
print("CACI Error:", mean_absolute_error(y_test_caci, caci_pred))


AQI Error: 13.914499782525898
CACI Error: 3.069943150303892


PART 11: Predict Future AQI
 - This selects the latest row (current pollution).
 - pred_aqi = aqi_model.predict(sample)

  pred_caci = caci_model.predict(sample) 
  
  AI predicts next hour pollution.


In [12]:
sample = X.iloc[-1:]   # latest row

pred_aqi = aqi_model.predict(sample)
pred_caci = caci_model.predict(sample)

print("Predicted AQI (1 hr):", pred_aqi[0])
print("Predicted CACI (1 hr):", pred_caci[0])


Predicted AQI (1 hr): 67.3287952658001
Predicted CACI (1 hr): 57.545577735149855


PART 12: Convert AQI to Status
 - Defines function.
 - Classifies pollution level.

In [13]:
def aqi_status(aqi):
    if aqi <= 50: return "Good"
    elif aqi <= 100: return "Satisfactory"
    elif aqi <= 200: return "Moderate"
    elif aqi <= 300: return "Poor"
    else: return "Severe"

print("Future AQI Status:", aqi_status(pred_aqi[0]))


Future AQI Status: Satisfactory


In [14]:
import joblib

joblib.dump(aqi_model, "aqi_model.pkl")
joblib.dump(caci_model, "caci_model.pkl")

print("Models saved ✅")


Models saved ✅


## PART 13: Better Training for Higher Accuracy

This section improves model performance using:
- time-series friendly split (no random leakage)
- lag features (previous AQI/CACI values)
- cyclical time encoding (hour/day patterns)
- hyperparameter tuning with cross-validation

In [15]:
import numpy as np

# Work on a copy so earlier cells stay unchanged
df2 = df.copy()

# Resolve expected columns case-insensitively (handles gasValue vs gasvalue, etc.)
column_lookup = {c.lower(): c for c in df2.columns}
required_columns = {
    'AQI': 'aqi',
    'CACI': 'caci',
    'PM2.5': 'pm2.5',
    'PM10': 'pm10',
    'TEMP': 'temp',
    'humidity': 'humidity',
    'gasValue': 'gasvalue'
}

missing = [name for name, key in required_columns.items() if key not in column_lookup]
if missing:
    raise KeyError(f"Missing required columns: {missing}. Available columns: {list(df2.columns)}")

# Create canonical columns so all later code uses one consistent naming scheme
for canonical_name, lookup_key in required_columns.items():
    actual_name = column_lookup[lookup_key]
    if actual_name != canonical_name:
        df2[canonical_name] = df2[actual_name]

# Add lag features: previous observations often improve next-step forecasting
for col in ['AQI', 'CACI', 'PM2.5', 'PM10', 'gasValue']:
    df2[f'{col}_lag1'] = df2[col].shift(1)
    df2[f'{col}_lag2'] = df2[col].shift(2)

# Cyclical encoding for hour/day patterns
if 'hour' not in df2.columns:
    df2['hour'] = df2['timestamp'].dt.hour
if 'day' not in df2.columns:
    df2['day'] = df2['timestamp'].dt.day
if 'weekday' not in df2.columns:
    df2['weekday'] = df2['timestamp'].dt.weekday

# Better target horizon (next 1 step) from current row
df2['AQI_next'] = df2['AQI'].shift(-1)
df2['CACI_next'] = df2['CACI'].shift(-1)

df2['hour_sin'] = np.sin(2 * np.pi * df2['hour'] / 24)
df2['hour_cos'] = np.cos(2 * np.pi * df2['hour'] / 24)
df2['weekday_sin'] = np.sin(2 * np.pi * df2['weekday'] / 7)
df2['weekday_cos'] = np.cos(2 * np.pi * df2['weekday'] / 7)

df2 = df2.dropna().reset_index(drop=True)

base_features = ['PM2.5', 'PM10', 'gasValue', 'TEMP', 'humidity']
lag_features = [
    'AQI_lag1', 'AQI_lag2', 'CACI_lag1', 'CACI_lag2',
    'PM2.5_lag1', 'PM2.5_lag2', 'PM10_lag1', 'PM10_lag2',
    'gasValue_lag1', 'gasValue_lag2'
 ]
time_features = ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'day']

features_v2 = base_features + lag_features + time_features
X2 = df2[features_v2]
y2_aqi = df2['AQI_next']
y2_caci = df2['CACI_next']

print('Improved dataset shape:', X2.shape)

Improved dataset shape: (31871, 20)


In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

# Time-series split: train on past, test on future
split_idx = int(len(X2) * 0.8)
X_train2, X_test2 = X2.iloc[:split_idx], X2.iloc[split_idx:]
y_train2_aqi, y_test2_aqi = y2_aqi.iloc[:split_idx], y2_aqi.iloc[split_idx:]
y_train2_caci, y_test2_caci = y2_caci.iloc[:split_idx], y2_caci.iloc[split_idx:]

tscv = TimeSeriesSplit(n_splits=5)

param_dist = {
    'n_estimators': [200, 300, 500, 700],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

base_rf = RandomForestRegressor(random_state=42, n_jobs=-1)

search_aqi = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_caci = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_aqi.fit(X_train2, y_train2_aqi)
search_caci.fit(X_train2, y_train2_caci)

aqi_model_best = search_aqi.best_estimator_
caci_model_best = search_caci.best_estimator_

pred2_aqi = aqi_model_best.predict(X_test2)
pred2_caci = caci_model_best.predict(X_test2)

aqi_mae = mean_absolute_error(y_test2_aqi, pred2_aqi)
caci_mae = mean_absolute_error(y_test2_caci, pred2_caci)
aqi_rmse = root_mean_squared_error(y_test2_aqi, pred2_aqi)
caci_rmse = root_mean_squared_error(y_test2_caci, pred2_caci)

print('Best AQI params:', search_aqi.best_params_)
print('Best CACI params:', search_caci.best_params_)
print('Improved AQI MAE :', round(aqi_mae, 3))
print('Improved AQI RMSE:', round(aqi_rmse, 3))
print('Improved CACI MAE :', round(caci_mae, 3))
print('Improved CACI RMSE:', round(caci_rmse, 3))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best AQI params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': None}
Best CACI params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': None}
Improved AQI MAE : 13.333
Improved AQI RMSE: 22.501
Improved CACI MAE : 3.288
Improved CACI RMSE: 4.636


In [22]:
# Use improved models as the default models for later prediction cells
aqi_model = aqi_model_best
caci_model = caci_model_best

import joblib
joblib.dump(aqi_model, 'aqi_model_1.pkl')
joblib.dump(caci_model, 'caci_model_1.pkl')

print('Improved models saved as aqi_model.pkl and caci_model.pkl')

Improved models saved as aqi_model.pkl and caci_model.pkl


## PART 14: Predict TEMP, Humidity, and PM2.5

This section uses the same improved feature set from Part 13 and predicts next-step values for:
- `TEMP_next`
- `humidity_next`
- `PM2.5_next`

In [23]:
# Create next-step targets
work_df = df2.copy()
work_df['TEMP_next'] = work_df['TEMP'].shift(-1)
work_df['humidity_next'] = work_df['humidity'].shift(-1)
work_df['PM2.5_next'] = work_df['PM2.5'].shift(-1)
work_df = work_df.dropna().reset_index(drop=True)

X_extra = work_df[features_v2]
y_temp = work_df['TEMP_next']
y_humidity = work_df['humidity_next']
y_pm25 = work_df['PM2.5_next']

split_idx_extra = int(len(X_extra) * 0.8)
X_train_e, X_test_e = X_extra.iloc[:split_idx_extra], X_extra.iloc[split_idx_extra:]
y_train_temp, y_test_temp = y_temp.iloc[:split_idx_extra], y_temp.iloc[split_idx_extra:]
y_train_hum, y_test_hum = y_humidity.iloc[:split_idx_extra], y_humidity.iloc[split_idx_extra:]
y_train_pm25, y_test_pm25 = y_pm25.iloc[:split_idx_extra], y_pm25.iloc[split_idx_extra:]

# Reuse tuned settings from AQI model for consistency and speed
temp_model = RandomForestRegressor(**search_aqi.best_params_, random_state=42, n_jobs=-1)
humidity_model = RandomForestRegressor(**search_aqi.best_params_, random_state=42, n_jobs=-1)
pm25_model = RandomForestRegressor(**search_aqi.best_params_, random_state=42, n_jobs=-1)

temp_model.fit(X_train_e, y_train_temp)
humidity_model.fit(X_train_e, y_train_hum)
pm25_model.fit(X_train_e, y_train_pm25)

pred_temp = temp_model.predict(X_test_e)
pred_hum = humidity_model.predict(X_test_e)
pred_pm25 = pm25_model.predict(X_test_e)

print('TEMP MAE :', round(mean_absolute_error(y_test_temp, pred_temp), 3))
print('TEMP RMSE:', round(root_mean_squared_error(y_test_temp, pred_temp), 3))
print('Humidity MAE :', round(mean_absolute_error(y_test_hum, pred_hum), 3))
print('Humidity RMSE:', round(root_mean_squared_error(y_test_hum, pred_hum), 3))
print('PM2.5 MAE :', round(mean_absolute_error(y_test_pm25, pred_pm25), 3))
print('PM2.5 RMSE:', round(root_mean_squared_error(y_test_pm25, pred_pm25), 3))

TEMP MAE : 0.671
TEMP RMSE: 0.945
Humidity MAE : 3.691
Humidity RMSE: 5.603
PM2.5 MAE : 10.308
PM2.5 RMSE: 19.091


In [19]:
# Predict next-step TEMP/Humidity/PM2.5 from the latest row
sample_extra = X_extra.iloc[-1:]

future_temp = temp_model.predict(sample_extra)[0]
future_humidity = humidity_model.predict(sample_extra)[0]
future_pm25 = pm25_model.predict(sample_extra)[0]

print('Predicted TEMP (next step):', round(future_temp, 2))
print('Predicted humidity (next step):', round(future_humidity, 2))
print('Predicted PM2.5 (next step):', round(future_pm25, 2))

joblib.dump(temp_model, 'temp_model.pkl')
joblib.dump(humidity_model, 'humidity_model.pkl')
joblib.dump(pm25_model, 'pm25_model.pkl')
print('Saved: temp_model.pkl, humidity_model.pkl, pm25_model.pkl')

Predicted TEMP (next step): 11.46
Predicted humidity (next step): 13.18
Predicted PM2.5 (next step): 15.45
Saved: temp_model.pkl, humidity_model.pkl, pm25_model.pkl


## PART 15: Deployment-Safe Models (No AQI/CACI Lag Dependency)

This section trains a second set of models for production use without `AQI_lag*` and `CACI_lag*`,
so realtime prediction does not depend on previously predicted AQI/CACI values.

In [20]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import joblib

# Reuse engineered table from Part 13
deploy_df = df2.copy()

# Production-safe feature set: remove AQI/CACI lag features
deploy_features = [
    'PM2.5', 'PM10', 'gasValue', 'TEMP', 'humidity',
    'PM2.5_lag1', 'PM2.5_lag2',
    'PM10_lag1', 'PM10_lag2',
    'gasValue_lag1', 'gasValue_lag2',
    'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'day'
 ]

X_deploy = deploy_df[deploy_features]
y_deploy_aqi = deploy_df['AQI_next']
y_deploy_caci = deploy_df['CACI_next']

split_idx_deploy = int(len(X_deploy) * 0.8)
X_train_d, X_test_d = X_deploy.iloc[:split_idx_deploy], X_deploy.iloc[split_idx_deploy:]
y_train_d_aqi, y_test_d_aqi = y_deploy_aqi.iloc[:split_idx_deploy], y_deploy_aqi.iloc[split_idx_deploy:]
y_train_d_caci, y_test_d_caci = y_deploy_caci.iloc[:split_idx_deploy], y_deploy_caci.iloc[split_idx_deploy:]

best_params_for_deploy = search_aqi.best_params_ if 'search_aqi' in globals() else {
    'n_estimators': 300,
    'max_depth': 20,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt'
}

aqi_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)
caci_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)

aqi_model_live.fit(X_train_d, y_train_d_aqi)
caci_model_live.fit(X_train_d, y_train_d_caci)

pred_d_aqi = aqi_model_live.predict(X_test_d)
pred_d_caci = caci_model_live.predict(X_test_d)

print('Live AQI MAE :', round(mean_absolute_error(y_test_d_aqi, pred_d_aqi), 3))
print('Live AQI RMSE:', round(root_mean_squared_error(y_test_d_aqi, pred_d_aqi), 3))
print('Live CACI MAE :', round(mean_absolute_error(y_test_d_caci, pred_d_caci), 3))
print('Live CACI RMSE:', round(root_mean_squared_error(y_test_d_caci, pred_d_caci), 3))

# Train deployment-safe TEMP/Humidity/PM2.5 models with same feature set
deploy_extra = deploy_df.copy()
deploy_extra['TEMP_next'] = deploy_extra['TEMP'].shift(-1)
deploy_extra['humidity_next'] = deploy_extra['humidity'].shift(-1)
deploy_extra['PM2.5_next'] = deploy_extra['PM2.5'].shift(-1)
deploy_extra = deploy_extra.dropna().reset_index(drop=True)

X_deploy_extra = deploy_extra[deploy_features]
y_deploy_temp = deploy_extra['TEMP_next']
y_deploy_humidity = deploy_extra['humidity_next']
y_deploy_pm25 = deploy_extra['PM2.5_next']

split_idx_deploy_extra = int(len(X_deploy_extra) * 0.8)
X_train_dx, X_test_dx = X_deploy_extra.iloc[:split_idx_deploy_extra], X_deploy_extra.iloc[split_idx_deploy_extra:]
y_train_temp_dx, y_test_temp_dx = y_deploy_temp.iloc[:split_idx_deploy_extra], y_deploy_temp.iloc[split_idx_deploy_extra:]
y_train_hum_dx, y_test_hum_dx = y_deploy_humidity.iloc[:split_idx_deploy_extra], y_deploy_humidity.iloc[split_idx_deploy_extra:]
y_train_pm_dx, y_test_pm_dx = y_deploy_pm25.iloc[:split_idx_deploy_extra], y_deploy_pm25.iloc[split_idx_deploy_extra:]

temp_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)
humidity_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)
pm25_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)

temp_model_live.fit(X_train_dx, y_train_temp_dx)
humidity_model_live.fit(X_train_dx, y_train_hum_dx)
pm25_model_live.fit(X_train_dx, y_train_pm_dx)

pred_temp_dx = temp_model_live.predict(X_test_dx)
pred_hum_dx = humidity_model_live.predict(X_test_dx)
pred_pm_dx = pm25_model_live.predict(X_test_dx)

print('Live TEMP MAE :', round(mean_absolute_error(y_test_temp_dx, pred_temp_dx), 3))
print('Live TEMP RMSE:', round(root_mean_squared_error(y_test_temp_dx, pred_temp_dx), 3))
print('Live HUM MAE  :', round(mean_absolute_error(y_test_hum_dx, pred_hum_dx), 3))
print('Live HUM RMSE :', round(root_mean_squared_error(y_test_hum_dx, pred_hum_dx), 3))
print('Live PM2.5 MAE :', round(mean_absolute_error(y_test_pm_dx, pred_pm_dx), 3))
print('Live PM2.5 RMSE:', round(root_mean_squared_error(y_test_pm_dx, pred_pm_dx), 3))

joblib.dump(aqi_model_live, 'aqi_model_live.pkl')
joblib.dump(caci_model_live, 'caci_model_live.pkl')
joblib.dump(temp_model_live, 'temp_model_live.pkl')
joblib.dump(humidity_model_live, 'humidity_model_live.pkl')
joblib.dump(pm25_model_live, 'pm25_model_live.pkl')

print('Saved deployment-safe models: aqi_model_live.pkl, caci_model_live.pkl, temp_model_live.pkl, humidity_model_live.pkl, pm25_model_live.pkl')

Live AQI MAE : 13.32
Live AQI RMSE: 22.51
Live CACI MAE : 3.108
Live CACI RMSE: 4.402
Live TEMP MAE : 0.676
Live TEMP RMSE: 0.948
Live HUM MAE  : 3.725
Live HUM RMSE : 5.645
Live PM2.5 MAE : 10.304
Live PM2.5 RMSE: 19.098
Saved deployment-safe models: aqi_model_live.pkl, caci_model_live.pkl, temp_model_live.pkl, humidity_model_live.pkl, pm25_model_live.pkl


## PART 17: India Local Adaptation (ThingSpeak History -> Retrain Live Models)

This block builds a local training table from your ThingSpeak history (2-minute data),
then retrains deployment-safe `*_live.pkl` models.

Note: AQI/CACI models are retrained only if ground-truth AQI/CACI labels are available in the input history file.

In [6]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed; using sklearn models.")

# ===== CONFIG =====
ADAPT_CHANNEL_ID = "3220962"
ADAPT_READ_API_KEY = "TF7VPOAMFV8XK33V"
ADAPT_RESULTS = 8000

ADAPT_SOURCE_MAP = {
    "field1": "TEMP",
    "field2": "humidity",
    "field3": "PM2.5",
    "field4": "PM10",
    "field5": "AQI",
    "field6": "CACI",
    "field7": "gasValue",
}

def load_data():
    url = (
        f"https://api.thingspeak.com/channels/{ADAPT_CHANNEL_ID}/feeds.csv"
        f"?api_key={ADAPT_READ_API_KEY}&results={ADAPT_RESULTS}"
    )
    return pd.read_csv(url)

def preprocess(raw):
    df = raw.rename(columns=ADAPT_SOURCE_MAP).copy()
    df["timestamp"] = pd.to_datetime(df["created_at"], errors="coerce")

    numeric_cols = ["TEMP", "humidity", "PM2.5", "PM10", "gasValue", "AQI", "CACI"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["AQI"] = df["AQI"].clip(lower=0, upper=df["AQI"].quantile(0.995))

    df = df.dropna(subset=["timestamp"] + numeric_cols)
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

def feature_engineering(df):
    d = df.copy()

    lag_steps = [1, 2, 3, 5, 10, 15, 30]
    for col in ["AQI", "CACI", "PM2.5", "PM10", "gasValue", "TEMP", "humidity"]:
        for lag in lag_steps:
            d[f"{col}_lag{lag}"] = d[col].shift(lag)

    for col in ["AQI", "PM2.5", "PM10", "gasValue", "CACI"]:
        for win in [5, 15, 30]:
            d[f"{col}_roll_mean_{win}"] = d[col].rolling(win).mean()
            d[f"{col}_roll_std_{win}"] = d[col].rolling(win).std()

    d["AQI_diff_1"] = d["AQI"] - d["AQI_lag1"]
    d["AQI_diff_5"] = d["AQI"] - d["AQI_lag5"]
    d["AQI_diff_15"] = d["AQI"] - d["AQI_lag15"]

    d["hour"] = d["timestamp"].dt.hour
    d["weekday"] = d["timestamp"].dt.weekday
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    d["weekday_sin"] = np.sin(2 * np.pi * d["weekday"] / 7)
    d["weekday_cos"] = np.cos(2 * np.pi * d["weekday"] / 7)

    horizon = 30
    d["AQI_next"] = d["AQI"].shift(-horizon)
    d["CACI_next"] = d["CACI"].shift(-horizon)
    d["TEMP_next"] = d["TEMP"].shift(-horizon)
    d["humidity_next"] = d["humidity"].shift(-horizon)
    d["PM2.5_next"] = d["PM2.5"].shift(-horizon)
    d["AQI_delta_next"] = d["AQI_next"] - d["AQI"]
    return d

def _predict_xgb(model, X):
    return np.asarray(model.predict(X), dtype=float)

def fit_best_aqi_model(X_train, y_train_abs, y_train_delta, aqi_train_now, X_val, y_val_abs, y_val_delta, aqi_val_now):
    models_abs = {
        "rf_abs": RandomForestRegressor(n_estimators=1000, max_depth=24, min_samples_leaf=2, random_state=42, n_jobs=-1),
        "hgb_abs": HistGradientBoostingRegressor(learning_rate=0.03, max_depth=10, max_iter=1000, min_samples_leaf=20, random_state=42),
        "extra_abs": ExtraTreesRegressor(n_estimators=1400, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1),
    }
    if HAS_XGBOOST:
        models_abs["xgb_abs"] = xgb.XGBRegressor(
            n_estimators=2200, learning_rate=0.018, max_depth=7, subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.2, reg_lambda=2.0, objective="reg:squarederror", random_state=42, n_jobs=-1
        )

    models_delta = {
        "rf_delta": RandomForestRegressor(n_estimators=1000, max_depth=24, min_samples_leaf=2, random_state=42, n_jobs=-1),
        "hgb_delta": HistGradientBoostingRegressor(learning_rate=0.03, max_depth=10, max_iter=1000, min_samples_leaf=20, random_state=42),
    }
    if HAS_XGBOOST:
        models_delta["xgb_delta"] = xgb.XGBRegressor(
            n_estimators=2200, learning_rate=0.018, max_depth=7, subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.2, reg_lambda=2.0, objective="reg:squarederror", random_state=42, n_jobs=-1
        )

    abs_preds = {}
    for name, m in models_abs.items():
        m.fit(X_train, y_train_abs)
        p = _predict_xgb(m, X_val) if "xgb" in name else m.predict(X_val)
        abs_preds[name] = p
        print(f"AQI abs candidate {name} MAE: {mean_absolute_error(y_val_abs, p):.4f}")

    delta_preds = {}
    delta_lo, delta_hi = np.quantile(y_train_delta, [0.01, 0.99])
    for name, m in models_delta.items():
        m.fit(X_train, y_train_delta)
        p_delta = _predict_xgb(m, X_val) if "xgb" in name else m.predict(X_val)
        p_delta = np.clip(p_delta, delta_lo, delta_hi)
        p_abs_from_delta = aqi_val_now.values + p_delta
        delta_preds[name] = p_abs_from_delta
        print(f"AQI delta candidate {name} MAE: {mean_absolute_error(y_val_abs, p_abs_from_delta):.4f}")

    best = {"mae": float("inf"), "abs_name": None, "delta_name": None, "wa": 1.0, "wd": 0.0}
    for abs_name, p_abs in abs_preds.items():
        for delta_name, p_delta_abs in delta_preds.items():
            for wa in [0.2, 0.3, 0.4, 0.5, 0.6]:
                for wd in [0.1, 0.2, 0.3, 0.4, 0.5]:
                    if wa + wd > 0.95:
                        continue
                    wp = 1.0 - wa - wd
                    p = wa * p_abs + wd * p_delta_abs + wp * aqi_val_now.values
                    mae = mean_absolute_error(y_val_abs, p)
                    if mae < best["mae"]:
                        best = {"mae": mae, "abs_name": abs_name, "delta_name": delta_name, "wa": wa, "wd": wd}

    print(
        "Selected AQI blend:",
        f"abs={best['abs_name']}, delta={best['delta_name']}, ",
        f"wa={best['wa']}, wd={best['wd']}, wp={1.0 - best['wa'] - best['wd']:.2f}, ",
        f"val_MAE={best['mae']:.4f}"
    )

    X_tv = pd.concat([X_train, X_val], axis=0)
    y_tv_abs = pd.concat([y_train_abs, y_val_abs], axis=0)
    y_tv_delta = pd.concat([y_train_delta, y_val_delta], axis=0)

    m_abs = models_abs[best["abs_name"]]
    m_delta = models_delta[best["delta_name"]]
    m_abs.fit(X_tv, y_tv_abs)
    m_delta.fit(X_tv, y_tv_delta)
    return m_abs, m_delta, best, delta_lo, delta_hi

def train_models():
    raw = load_data()
    df = preprocess(raw)
    df = feature_engineering(df).dropna().reset_index(drop=True)

    aqi_features = [
        "AQI", "CACI", "PM2.5", "PM10", "gasValue", "TEMP", "humidity",
        "AQI_lag1", "AQI_lag2", "AQI_lag3", "AQI_lag5", "AQI_lag10", "AQI_lag15", "AQI_lag30",
        "CACI_lag1", "CACI_lag5", "CACI_lag15", "CACI_lag30",
        "PM2.5_lag1", "PM2.5_lag5", "PM2.5_lag15", "PM2.5_lag30",
        "PM10_lag1", "PM10_lag5", "PM10_lag15", "PM10_lag30",
        "gasValue_lag1", "gasValue_lag5", "gasValue_lag15", "gasValue_lag30",
        "AQI_roll_mean_5", "AQI_roll_mean_15", "AQI_roll_mean_30",
        "AQI_roll_std_5", "AQI_roll_std_15", "AQI_roll_std_30",
        "PM2.5_roll_mean_15", "PM2.5_roll_std_15",
        "PM10_roll_mean_15", "PM10_roll_std_15",
        "gasValue_roll_mean_15", "gasValue_roll_std_15",
        "AQI_diff_1", "AQI_diff_5", "AQI_diff_15",
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos"
    ]

    base_features = [
        "PM2.5", "PM10", "gasValue", "TEMP", "humidity",
        "PM2.5_lag1", "PM2.5_lag5", "PM2.5_lag15",
        "PM10_lag1", "PM10_lag5", "PM10_lag15",
        "gasValue_lag1", "gasValue_lag5", "gasValue_lag15",
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos"
    ]

    n = len(df)
    train_end = int(n * 0.75)
    val_end = int(n * 0.90)

    X_train = df[aqi_features].iloc[:train_end]
    X_val = df[aqi_features].iloc[train_end:val_end]
    X_test = df[aqi_features].iloc[val_end:]

    y_train_abs = df["AQI_next"].iloc[:train_end]
    y_val_abs = df["AQI_next"].iloc[train_end:val_end]
    y_test_abs = df["AQI_next"].iloc[val_end:]

    y_train_delta = df["AQI_delta_next"].iloc[:train_end]
    y_val_delta = df["AQI_delta_next"].iloc[train_end:val_end]

    aqi_train_now = df["AQI"].iloc[:train_end]
    aqi_val_now = df["AQI"].iloc[train_end:val_end]
    aqi_test_now = df["AQI"].iloc[val_end:]

    print("Training AQI with abs+delta ensemble selection...")
    aqi_abs_model, aqi_delta_model, blend_cfg, delta_lo, delta_hi = fit_best_aqi_model(
        X_train, y_train_abs, y_train_delta, aqi_train_now,
        X_val, y_val_abs, y_val_delta, aqi_val_now
    )

    p_abs = _predict_xgb(aqi_abs_model, X_test) if HAS_XGBOOST and aqi_abs_model.__class__.__name__.startswith("XGB") else aqi_abs_model.predict(X_test)
    p_delta = _predict_xgb(aqi_delta_model, X_test) if HAS_XGBOOST and aqi_delta_model.__class__.__name__.startswith("XGB") else aqi_delta_model.predict(X_test)
    p_delta = np.clip(p_delta, delta_lo, delta_hi)
    p_delta_abs = aqi_test_now.values + p_delta
    wp = 1.0 - blend_cfg["wa"] - blend_cfg["wd"]
    p_final = blend_cfg["wa"] * p_abs + blend_cfg["wd"] * p_delta_abs + wp * aqi_test_now.values
    p_final = np.clip(p_final, 0, None)
    print("AQI MAE:", mean_absolute_error(y_test_abs.values, p_final))

    Xb = df[base_features]
    split = int(n * 0.8)
    Xb_train = Xb.iloc[:split]
    Xb_test = Xb.iloc[split:]

    print("Training TEMP...")
    temp_model = RandomForestRegressor(n_estimators=450, max_depth=20, min_samples_leaf=2, random_state=42, n_jobs=-1)
    temp_model.fit(Xb_train, df["TEMP_next"].iloc[:split])
    temp_pred = temp_model.predict(Xb_test)
    print("TEMP MAE:", mean_absolute_error(df["TEMP_next"].iloc[split:], temp_pred))

    print("Training HUM...")
    hum_model = RandomForestRegressor(n_estimators=450, max_depth=20, min_samples_leaf=2, random_state=42, n_jobs=-1)
    hum_model.fit(Xb_train, df["humidity_next"].iloc[:split])
    hum_pred = hum_model.predict(Xb_test)
    print("HUM MAE:", mean_absolute_error(df["humidity_next"].iloc[split:], hum_pred))

    print("Training PM2.5...")
    if HAS_XGBOOST:
        pm_model = xgb.XGBRegressor(
            n_estimators=1000, learning_rate=0.025, max_depth=6, subsample=0.9,
            colsample_bytree=0.9, objective="reg:squarederror", random_state=42, n_jobs=-1
        )
    else:
        pm_model = ExtraTreesRegressor(n_estimators=900, random_state=42, n_jobs=-1)
    pm_model.fit(Xb_train, df["PM2.5_next"].iloc[:split])
    pm_pred = _predict_xgb(pm_model, Xb_test) if HAS_XGBOOST else pm_model.predict(Xb_test)
    print("PM2.5 MAE:", mean_absolute_error(df["PM2.5_next"].iloc[split:], pm_pred))

    print("Training CACI...")
    caci_model = RandomForestRegressor(n_estimators=500, max_depth=22, min_samples_leaf=2, random_state=42, n_jobs=-1)
    caci_model.fit(Xb_train, df["CACI_next"].iloc[:split])
    caci_pred = caci_model.predict(Xb_test)
    print("CACI MAE:", mean_absolute_error(df["CACI_next"].iloc[split:], caci_pred))

    joblib.dump(aqi_abs_model, "aqi_model_live.pkl")
    joblib.dump(caci_model, "caci_model_live.pkl")
    joblib.dump(temp_model, "temp_model_live.pkl")
    joblib.dump(hum_model, "humidity_model_live.pkl")
    joblib.dump(pm_model, "pm25_model_live.pkl")

    print("\nALL MODELS TRAINED & SAVED SUCCESSFULLY")

train_models()

Training AQI with abs+delta ensemble selection...
AQI abs candidate rf_abs MAE: 41.4727
AQI abs candidate hgb_abs MAE: 36.0982
AQI abs candidate extra_abs MAE: 38.6823
AQI abs candidate xgb_abs MAE: 37.3477
AQI delta candidate rf_delta MAE: 26.3405
AQI delta candidate hgb_delta MAE: 26.9663
AQI delta candidate xgb_delta MAE: 25.4388
Selected AQI blend: abs=hgb_abs, delta=xgb_delta,  wa=0.2, wd=0.1, wp=0.70,  val_MAE=18.9982
AQI MAE: 18.23827476288828
Training TEMP...
TEMP MAE: 0.846911519175271
Training HUM...
HUM MAE: 6.549028902197959
Training PM2.5...
PM2.5 MAE: 16.965886101020082
Training CACI...
CACI MAE: 13.627542457535126

ALL MODELS TRAINED & SAVED SUCCESSFULLY
